# NGIML Inference

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/juhenes/ngiml"
REPO_BRANCH = "FurtherEnhancement"
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
print(f"Repo ready at {REPO_DIR} on branch {REPO_BRANCH}")

In [ ]:
from __future__ import annotations

import io
import json
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CHECKPOINT_PATH = Path('/content/drive/MyDrive/ngiml-casia-enhanced/checkpoints/best_checkpoint.pt')
HF_DATASET_REPO_ID = 'juhenes/ngiml-test'
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/ngiml-casia-inference')
HF_SNAPSHOT_LOCAL_DIR = Path('/content/hf_datasets/ngiml_test')

INFERENCE_STRATEGY = 'direct'
THRESHOLD_FOR_METRICS = None
PLOT_BINARY_THRESHOLD = 0.5
DIRECT_BATCH_SIZE = 8

CSV_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'csv'
PLOT_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / 'plots'
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents, Path('/content/ngiml')]:
        if (p / 'tools' / 'infer_helpers.py').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('Could not find NGIML repo root.')

REPO_ROOT = find_repo_root()
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.infer_helpers import load_model_from_checkpoint, predict_probability_map_by_strategy, predict_probability_maps_batch, resolve_normalization_mode_for_inference

assert CHECKPOINT_PATH.exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
model, device, ckpt_info = load_model_from_checkpoint(CHECKPOINT_PATH)
normalization_mode = resolve_normalization_mode_for_inference(checkpoint_path=CHECKPOINT_PATH, default_mode='imagenet')
threshold_used = float(ckpt_info.get('default_threshold', 0.5) if THRESHOLD_FOR_METRICS is None else THRESHOLD_FOR_METRICS)
print('Device:', device)
print('Normalization:', normalization_mode)
print('Threshold for CSV metrics:', threshold_used)
print('Plot threshold:', PLOT_BINARY_THRESHOLD)
print('Direct batch size:', DIRECT_BATCH_SIZE if INFERENCE_STRATEGY == 'direct' else 'n/a')

In [ ]:
def _to_chw_rgb(image_np: np.ndarray) -> np.ndarray:
    if image_np.ndim == 2:
        image_np = np.stack([image_np, image_np, image_np], axis=-1)
    if image_np.ndim == 3 and image_np.shape[0] in (1, 3) and image_np.shape[-1] not in (1, 3):
        image_np = np.transpose(image_np, (1, 2, 0))
    if image_np.ndim != 3:
        raise ValueError(f'Unsupported image shape: {image_np.shape}')
    if image_np.shape[-1] == 1:
        image_np = np.repeat(image_np, 3, axis=-1)
    if image_np.shape[-1] > 3:
        image_np = image_np[..., :3]
    return np.transpose(image_np, (2, 0, 1))

def _to_hw_mask(mask_np: np.ndarray | None, h: int, w: int) -> np.ndarray:
    if mask_np is None:
        return np.zeros((h, w), dtype=np.uint8)
    arr = np.asarray(mask_np)
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[..., 0]
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = (arr > 0).astype(np.uint8)
    if arr.shape != (h, w):
        raise ValueError(f'Mask shape {arr.shape} does not match image {(h, w)}')
    return arr

def _parse_meta(raw) -> dict:
    if raw is None:
        return {}
    if isinstance(raw, np.ndarray):
        raw = raw.item()
    if isinstance(raw, bytes):
        raw = raw.decode('utf-8', errors='replace')
    if isinstance(raw, str):
        try:
            return json.loads(raw)
        except Exception:
            return {'metadata_raw': raw}
    return raw if isinstance(raw, dict) else {'metadata_raw': str(raw)}

def _dataset_name(sample_uri: str, meta: dict) -> str:
    for k in ('dataset', 'dataset_name', 'source_dataset'):
        if str(meta.get(k, '')).strip():
            return str(meta[k]).strip()
    parts = Path(sample_uri.split('::')[0]).parts
    for i, part in enumerate(parts):
        if part.lower() in {'test', 'val', 'train'} and i > 0:
            return parts[i - 1]
    return parts[0] if parts else 'unknown'

def iter_prepared_samples(snapshot_root: Path):
    for npz_path in sorted(snapshot_root.rglob('*.npz')):
        with np.load(npz_path, allow_pickle=True) as blob:
            yield str(npz_path), {k: blob[k] for k in blob.files}

    for tar_path in sorted(snapshot_root.rglob('*.tar')):
        with tarfile.open(tar_path, mode='r') as tf:
            for m in tf.getmembers():
                if not m.isfile() or not m.name.lower().endswith('.npz'):
                    continue
                fobj = tf.extractfile(m)
                if fobj is None:
                    continue
                with np.load(io.BytesIO(fobj.read()), allow_pickle=True) as blob:
                    yield f'{tar_path}::{m.name}', {k: blob[k] for k in blob.files}

def compute_binary_metrics(pred_bin: np.ndarray, gt_bin: np.ndarray) -> dict[str, float]:
    pred = pred_bin.astype(bool)
    gt = gt_bin.astype(bool)
    tp = float(np.logical_and(pred, gt).sum())
    tn = float(np.logical_and(~pred, ~gt).sum())
    fp = float(np.logical_and(pred, ~gt).sum())
    fn = float(np.logical_and(~pred, gt).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 1.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 1.0
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 1.0
    return {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn, 'precision': precision, 'recall': recall, 'f1': f1, 'iou': iou, 'accuracy': acc}

def save_sample_plot(out_path: Path, image_chw: np.ndarray, gt_hw: np.ndarray, prob_hw: np.ndarray, bin05_hw: np.ndarray, title: str):
    img = np.transpose(image_chw, (1, 2, 0)).astype(np.float32)
    if img.max() > 1.0:
        img = img / 255.0
    img = np.clip(img, 0.0, 1.0)
    alpha = 0.45 * prob_hw[..., None].astype(np.float32)
    overlay = np.clip(img * (1.0 - alpha) + np.array([1.0, 0.0, 0.0]) * alpha, 0.0, 1.0)

    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    fig.suptitle(title, fontsize=11)
    axes[0].imshow(img); axes[0].set_title('Image'); axes[0].axis('off')
    axes[1].imshow(gt_hw, cmap='gray', vmin=0, vmax=1); axes[1].set_title('Ground Truth'); axes[1].axis('off')
    axes[2].imshow(prob_hw, cmap='magma', vmin=0, vmax=1); axes[2].set_title('Pred Mask (magma)'); axes[2].axis('off')
    axes[3].imshow(bin05_hw, cmap='gray', vmin=0, vmax=1); axes[3].set_title('Pred Mask (0.5 threshold)'); axes[3].axis('off')
    axes[4].imshow(overlay); axes[4].set_title('Overlay'); axes[4].axis('off')
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.close(fig)

In [ ]:
npz_files = list(HF_SNAPSHOT_LOCAL_DIR.rglob('*.npz'))
tar_files = list(HF_SNAPSHOT_LOCAL_DIR.rglob('*.tar'))
total_samples = len(npz_files) + sum(
    len([m for m in tarfile.open(tar_path, mode='r').getmembers() if m.isfile() and m.name.lower().endswith('.npz')])
    for tar_path in tar_files
)

snapshot_path = Path(snapshot_download(repo_id=HF_DATASET_REPO_ID, repo_type='dataset', local_dir=str(HF_SNAPSHOT_LOCAL_DIR), local_dir_use_symlinks=False))
print('Snapshot:', snapshot_path)

rows = []
plot_samples: dict[str, list[dict]] = {}

def _append_result(rows: list[dict], plot_samples: dict[str, list[dict]], sample: dict, prob: torch.Tensor) -> None:
    prob_hw = prob.detach().cpu().numpy().astype(np.float32)
    pred_bin_metric = (prob_hw >= float(threshold_used)).astype(np.uint8)
    pred_bin_05 = (prob_hw >= float(PLOT_BINARY_THRESHOLD)).astype(np.uint8)
    m = compute_binary_metrics(pred_bin_metric, sample['mask_hw'])

    raw_label = sample['meta'].get('label', int(sample['mask_hw'].max() > 0))
    if isinstance(raw_label, str):
        sample_label = 1 if raw_label.strip().lower() in {'1', 'fake', 'tp', 'tampered', 'manipulated'} else 0
    else:
        sample_label = int(raw_label)

    row = {
        'dataset': sample['dataset'], 'sample_uri': sample['sample_uri'], 'split': str(sample['meta'].get('split', 'test')),
        'label': sample_label, 'strategy': INFERENCE_STRATEGY,
        'normalization_mode': normalization_mode, 'threshold_for_metrics': float(threshold_used),
        'plot_binary_threshold': float(PLOT_BINARY_THRESHOLD), 'height': sample['h'], 'width': sample['w'],
        'mean_probability': float(prob_hw.mean()), 'max_probability': float(prob_hw.max()),
        'pred_positive_ratio_threshold': float(pred_bin_metric.mean()),
        'pred_positive_ratio_0_5': float(pred_bin_05.mean()), 'gt_positive_ratio': float(sample['mask_hw'].mean())
    }
    row.update(m)
    rows.append(row)

    if sample_label == 1:
        ds_bucket = plot_samples.setdefault(sample['dataset'], [])
        if len(ds_bucket) < 5:
            ds_bucket.append({'sample_uri': sample['sample_uri'], 'image_chw': sample['image_chw'], 'mask_hw': sample['mask_hw'], 'prob_hw': prob_hw, 'bin05_hw': pred_bin_05})

def _flush_direct_batch(rows: list[dict], plot_samples: dict[str, list[dict]], pending: list[dict]) -> None:
    if not pending:
        return
    probs = predict_probability_maps_batch(
        model=model,
        images=[sample['image_t'] for sample in pending],
        device=device,
        normalization_mode=normalization_mode,
    )
    for sample, prob in zip(pending, probs):
        _append_result(rows, plot_samples, sample, prob.clamp(0.0, 1.0))

pending_direct: dict[tuple[int, int], list[dict]] = {}
direct_batch_size = max(1, int(DIRECT_BATCH_SIZE)) if INFERENCE_STRATEGY == 'direct' else 1

for sample_uri, data in tqdm(iter_prepared_samples(snapshot_path), desc='Inference', total=total_samples):
    if 'image' not in data:
        continue

    image_chw = _to_chw_rgb(np.asarray(data['image']))
    h, w = int(image_chw.shape[1]), int(image_chw.shape[2])
    mask_hw = _to_hw_mask(data.get('mask'), h, w)
    meta = _parse_meta(data.get('metadata_json'))
    dataset = _dataset_name(sample_uri, meta)

    image_t = torch.from_numpy(image_chw).float()
    if image_t.max() > 1.0:
        image_t = image_t / 255.0

    sample = {
        'sample_uri': sample_uri,
        'image_chw': image_chw,
        'image_t': image_t,
        'mask_hw': mask_hw,
        'meta': meta,
        'dataset': dataset,
        'h': h,
        'w': w,
    }

    if INFERENCE_STRATEGY == 'direct':
        key = (h, w)
        batch = pending_direct.setdefault(key, [])
        batch.append(sample)
        if len(batch) >= direct_batch_size:
            _flush_direct_batch(rows, plot_samples, batch)
            batch.clear()
        continue

    prob = predict_probability_map_by_strategy(
        model=model,
        image=image_t,
        device=device,
        strategy=INFERENCE_STRATEGY,
        normalization_mode=normalization_mode,
    ).clamp(0.0, 1.0)
    _append_result(rows, plot_samples, sample, prob)

for pending in pending_direct.values():
    _flush_direct_batch(rows, plot_samples, pending)

results_df = pd.DataFrame(rows).sort_values(['dataset', 'sample_uri']).reset_index(drop=True)
if results_df.empty:
    raise RuntimeError('No samples processed from HF snapshot.')

results_csv = CSV_OUTPUT_DIR / 'ngiml_hf_test_inference_results.csv'
summary_csv = CSV_OUTPUT_DIR / 'ngiml_hf_test_inference_summary_by_dataset.csv'
results_df.to_csv(results_csv, index=False)

summary_df = results_df.groupby('dataset', as_index=False).agg({
    'sample_uri': 'count', 'f1': 'mean', 'iou': 'mean', 'precision': 'mean', 'recall': 'mean',
    'accuracy': 'mean', 'mean_probability': 'mean', 'pred_positive_ratio_threshold': 'mean', 'gt_positive_ratio': 'mean'
}).rename(columns={'sample_uri': 'num_samples'})
summary_df.to_csv(summary_csv, index=False)

for ds_name, samples in sorted(plot_samples.items()):
    ds_dir = PLOT_OUTPUT_DIR / ds_name
    for i, sample in enumerate(samples, start=1):
        out_png = ds_dir / f'{ds_name}_sample_{i:02d}.png'
        title = f"{ds_name} | sample {i} | {Path(sample['sample_uri'].split('::')[0]).name}"
        save_sample_plot(out_png, sample['image_chw'], sample['mask_hw'], sample['prob_hw'], sample['bin05_hw'], title)

print('Saved full CSV:', results_csv)
print('Saved summary CSV:', summary_csv)
print('Saved plot root:', PLOT_OUTPUT_DIR)
display(summary_df)

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

from tools.infer_helpers import load_model_from_checkpoint

try:
    from thop import clever_format, profile
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "thop"])
    from thop import clever_format, profile

checkpoint_dict_content = torch.load(CHECKPOINT_PATH, map_location="cpu")
training_config = checkpoint_dict_content.get("training_config", {})

model, _, ckpt_info = load_model_from_checkpoint(CHECKPOINT_PATH)
model = model.cpu().eval()

input_size = int(training_config.get("input_size", 448))

class _NgimlForProfiling(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        out = self.base_model(x, target_size=x.shape[-2:], residual_noise=None)
        if isinstance(out, (list, tuple)):
            return out[0]
        return out

wrapper = _NgimlForProfiling(model).eval()
dummy = torch.randn(1, 3, input_size, input_size, dtype=torch.float32)

with torch.no_grad():
    macs, params = profile(wrapper, inputs=(dummy,), verbose=False)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
flops = 2.0 * macs

macs_hr, params_hr = clever_format([macs, params], "%.3f")
flops_hr = clever_format([flops], "%.3f")

print("Checkpoint:", CHECKPOINT_PATH)
print("Input shape:", tuple(dummy.shape))
print("Trainable params:", f"{trainable_params:,}")
print("Total params:", f"{total_params:,}")
print("THOP params:", params_hr)
print("MACs:", macs_hr)
print("Approx FLOPs (2 * MACs):", flops_hr)